<div dir="rtl" style="text-align: center; padding: 30px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 15px;">

# 🎯 אלגוריתם KNN
## K-Nearest Neighbors

### מחברת 6 | קורס מדעי הנתונים

**משך:** 4-5 שעות

---

**מה נלמד?**
- הרעיון מאחורי KNN
- חישוב מרחק בין נקודות
- בחירת K אופטימלי
- נרמול נתונים ולמה זה קריטי
- חלוקה ל-Train/Test
- מדידת ביצועים

</div>

<div dir="rtl" style="text-align: right; background-color: #fff3e0; padding: 20px; border-radius: 15px; border: 3px solid #ff9800; margin-bottom: 20px;">

# 📖 הוראות שימוש במחברת

## סוגי התאים במחברת:

| סמל | סוג | מה לעשות |
|-----|-----|----------|
| 🎨 | **תא הדגמה** | רק ללחוץ ▶ להרצה. **אין צורך להבין את הקוד!** |
| ✍️ | **תא תרגול** | לכתוב קוד בעצמכם |
| 📝 | **הסבר** | לקרוא ולהבין |

## ⚠️ חשוב!
- **הריצו את כל תאי ההדגמה** (🎨) כדי לראות את הגרפים
- הקוד בתאי ההדגמה **מוסתר** - לחצו על החץ משמאל אם רוצים לראות אותו
- התמקדו בתאי התרגול (✍️) - שם תכתבו קוד

</div>

<div dir="rtl" style="text-align: right;">

# חלק 1: הרעיון מאחורי KNN 🧠

---

</div>

<div dir="rtl" style="text-align: right; background-color: #e3f2fd; padding: 20px; border-radius: 10px; border-right: 4px solid #2196f3;">

## 📚 מהו KNN?

**KNN (K-Nearest Neighbors)** הוא אלגוריתם פשוט וחזק לסיווג.

### הרעיון בפשטות:
> **"ספר לי מי החברים שלך - ואגיד לך מי אתה"** 👥

כדי לסווג נקודה חדשה:
1. מוצאים את **K השכנים הקרובים** אליה
2. בודקים לאיזו קטגוריה **רוב השכנים** שייכים
3. מסווגים את הנקודה החדשה לאותה קטגוריה

</div>

In [ ]:
#@title 🎨 **הדגמה** - לחצו ▶ להרצה (הקוד מוסתר - אין צורך להבין אותו)
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

print('Libraries imported successfully!')

<div dir="rtl" style="text-align: right;">

## 🎨 ויזואליזציה של הרעיון

נראה איך KNN עובד על דוגמה פשוטה:

</div>

In [ ]:
#@title 🎨 **הדגמה** - לחצו ▶ להרצה (הקוד מוסתר - אין צורך להבין אותו)
# Create simple example data
# Class A (blue) - cluster around (2,2)
class_a_x = np.random.normal(2, 0.5, 10)
class_a_y = np.random.normal(2, 0.5, 10)

# Class B (red) - cluster around (5,5)
class_b_x = np.random.normal(5, 0.5, 10)
class_b_y = np.random.normal(5, 0.5, 10)

# New point to classify
new_point = (3.5, 3.5)

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, k in zip(axes, [1, 3, 5]):
    # Plot existing points
    ax.scatter(class_a_x, class_a_y, c='#3498db', s=100, label='Class A', edgecolor='white')
    ax.scatter(class_b_x, class_b_y, c='#e74c3c', s=100, label='Class B', edgecolor='white')
    
    # Plot new point
    ax.scatter(new_point[0], new_point[1], c='#2ecc71', s=200, marker='*', 
               label='New point', edgecolor='black', linewidth=2)
    
    # Calculate distances to all points
    all_x = np.concatenate([class_a_x, class_b_x])
    all_y = np.concatenate([class_a_y, class_b_y])
    all_labels = ['A']*10 + ['B']*10
    
    distances = np.sqrt((all_x - new_point[0])**2 + (all_y - new_point[1])**2)
    nearest_idx = np.argsort(distances)[:k]
    
    # Draw circles to nearest neighbors
    for idx in nearest_idx:
        color = '#3498db' if all_labels[idx] == 'A' else '#e74c3c'
        ax.plot([new_point[0], all_x[idx]], [new_point[1], all_y[idx]], 
                color=color, linestyle='--', alpha=0.7, linewidth=2)
    
    # Count votes
    votes_a = sum(1 for idx in nearest_idx if all_labels[idx] == 'A')
    votes_b = k - votes_a
    prediction = 'A' if votes_a > votes_b else 'B'
    pred_color = '#3498db' if prediction == 'A' else '#e74c3c'
    
    ax.set_title(f'K = {k}\nVotes: A={votes_a}, B={votes_b}\nPrediction: Class {prediction}', 
                fontsize=12, color=pred_color)
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    ax.legend(loc='upper left')
    ax.set_xlim(0, 7)
    ax.set_ylim(0, 7)

plt.tight_layout()
plt.show()

<div dir="rtl" style="text-align: right; background-color: #e8f5e9; padding: 15px; border-radius: 10px; border-right: 4px solid #4caf50;">

### 💡 מה למדנו מהגרף?

- **K=1:** מסתכלים רק על השכן הכי קרוב - יכול להיות רגיש לרעש
- **K=3:** מסתכלים על 3 שכנים - יותר יציב
- **K=5:** מסתכלים על 5 שכנים - עוד יותר יציב
- ערך K **משפיע על התוצאה!**

</div>

<div dir="rtl" style="text-align: right;">

# חלק 2: חישוב מרחק 📏

---

כדי למצוא את השכנים הקרובים, צריך לחשב **מרחק** בין נקודות.

</div>

<div dir="rtl" style="text-align: right; background-color: #fff3e0; padding: 20px; border-radius: 10px; border-right: 4px solid #ff9800;">

## 📐 מרחק אוקלידי (Euclidean Distance)

המרחק הכי נפוץ - "קו ישר" בין שתי נקודות.

### הנוסחה (עבור 2 ממדים):

$$d = \sqrt{(x_2 - x_1)^2 + (y_2 - y_1)^2}$$

### דוגמה:
נקודה A: (1, 2)

נקודה B: (4, 6)

$$d = \sqrt{(4-1)^2 + (6-2)^2} = \sqrt{9 + 16} = \sqrt{25} = 5$$

</div>

In [ ]:
#@title 🎨 **הדגמה** - לחצו ▶ להרצה (הקוד מוסתר - אין צורך להבין אותו)
# Calculate Euclidean distance manually
def euclidean_distance(point1, point2):
    """Calculate Euclidean distance between two points"""
    return np.sqrt(sum((a - b) ** 2 for a, b in zip(point1, point2)))

# Example
point_a = (1, 2)
point_b = (4, 6)

distance = euclidean_distance(point_a, point_b)
print(f'Point A: {point_a}')
print(f'Point B: {point_b}')
print(f'Distance: {distance}')

In [ ]:
#@title 🎨 **הדגמה** - לחצו ▶ להרצה (הקוד מוסתר - אין צורך להבין אותו)
# Visualize the distance
fig, ax = plt.subplots(figsize=(8, 6))

# Plot points
ax.scatter(*point_a, s=200, c='#3498db', zorder=3, label=f'A {point_a}')
ax.scatter(*point_b, s=200, c='#e74c3c', zorder=3, label=f'B {point_b}')

# Draw the distance line
ax.plot([point_a[0], point_b[0]], [point_a[1], point_b[1]], 
        'g-', linewidth=3, label=f'Distance = {distance:.1f}')

# Draw the right triangle
ax.plot([point_a[0], point_b[0]], [point_a[1], point_a[1]], 'b--', alpha=0.5)
ax.plot([point_b[0], point_b[0]], [point_a[1], point_b[1]], 'r--', alpha=0.5)

# Annotations
ax.annotate(f'Δx = {point_b[0] - point_a[0]}', 
            xy=((point_a[0] + point_b[0])/2, point_a[1] - 0.3), fontsize=12, color='blue')
ax.annotate(f'Δy = {point_b[1] - point_a[1]}', 
            xy=(point_b[0] + 0.2, (point_a[1] + point_b[1])/2), fontsize=12, color='red')

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_title('Euclidean Distance = √(Δx² + Δy²)', fontsize=14)
ax.legend()
ax.set_xlim(-1, 7)
ax.set_ylim(-1, 8)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.show()

<div dir="rtl" style="text-align: right;">

# חלק 3: נרמול נתונים - למה זה קריטי? ⚖️

---

</div>

<div dir="rtl" style="text-align: right; background-color: #ffebee; padding: 20px; border-radius: 10px; border-right: 4px solid #f44336;">

## ⚠️ הבעיה: סקאלות שונות

נניח שיש לנו שני מאפיינים:
- **גיל:** 20-80 שנים
- **משכורת:** 5,000-50,000 ש"ח

**מה קורה כשמחשבים מרחק?**

הפרש של 10,000 ש"ח במשכורת "שווה" יותר מהפרש של 30 שנים בגיל!

**התוצאה:** המשכורת תשלוט לחלוטין בחישוב המרחק, והגיל יתעלם.

</div>

In [ ]:
#@title 🎨 **הדגמה** - לחצו ▶ להרצה (הקוד מוסתר - אין צורך להבין אותו)
# Demonstrate the problem with different scales
# Without normalization
person_a = {'age': 25, 'salary': 8000}
person_b = {'age': 55, 'salary': 8500}  # Same-ish salary, different age
person_c = {'age': 26, 'salary': 45000}  # Same-ish age, different salary

# Calculate distances from A
dist_ab = euclidean_distance([person_a['age'], person_a['salary']], 
                              [person_b['age'], person_b['salary']])
dist_ac = euclidean_distance([person_a['age'], person_a['salary']], 
                              [person_c['age'], person_c['salary']])

print('WITHOUT NORMALIZATION:')
print(f'Person A: age={person_a["age"]}, salary={person_a["salary"]}')
print(f'Person B: age={person_b["age"]}, salary={person_b["salary"]}')
print(f'Person C: age={person_c["age"]}, salary={person_c["salary"]}')
print(f'\nDistance A→B: {dist_ab:.1f}')
print(f'Distance A→C: {dist_ac:.1f}')
print(f'\n❌ B is "closer" even though age difference is 30 years!')

<div dir="rtl" style="text-align: right; background-color: #e8f5e9; padding: 20px; border-radius: 10px; border-right: 4px solid #4caf50;">

## ✅ הפתרון: נרמול (Normalization)

### Min-Max Normalization:
ממירים את כל הערכים לטווח 0-1:

$$x_{norm} = \frac{x - x_{min}}{x_{max} - x_{min}}$$

### StandardScaler (Z-score):
ממירים לממוצע 0 וסטיית תקן 1:

$$x_{norm} = \frac{x - \mu}{\sigma}$$

</div>

In [ ]:
#@title 🎨 **הדגמה** - לחצו ▶ להרצה (הקוד מוסתר - אין צורך להבין אותו)
# With normalization (Min-Max to 0-1)
age_min, age_max = 20, 80
salary_min, salary_max = 5000, 50000

def normalize(value, min_val, max_val):
    return (value - min_val) / (max_val - min_val)

# Normalize all values
a_norm = [normalize(person_a['age'], age_min, age_max), 
          normalize(person_a['salary'], salary_min, salary_max)]
b_norm = [normalize(person_b['age'], age_min, age_max), 
          normalize(person_b['salary'], salary_min, salary_max)]
c_norm = [normalize(person_c['age'], age_min, age_max), 
          normalize(person_c['salary'], salary_min, salary_max)]

dist_ab_norm = euclidean_distance(a_norm, b_norm)
dist_ac_norm = euclidean_distance(a_norm, c_norm)

print('WITH NORMALIZATION (0-1):')
print(f'Person A normalized: age={a_norm[0]:.2f}, salary={a_norm[1]:.2f}')
print(f'Person B normalized: age={b_norm[0]:.2f}, salary={b_norm[1]:.2f}')
print(f'Person C normalized: age={c_norm[0]:.2f}, salary={c_norm[1]:.2f}')
print(f'\nDistance A→B: {dist_ab_norm:.3f}')
print(f'Distance A→C: {dist_ac_norm:.3f}')
print(f'\n✅ Now C is closer (same age group)!')

In [ ]:
#@title 🎨 **הדגמה** - לחצו ▶ להרצה (הקוד מוסתר - אין צורך להבין אותו)
# Visualize the difference
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Without normalization
axes[0].scatter(person_a['age'], person_a['salary'], s=200, c='#3498db', label='A', zorder=3)
axes[0].scatter(person_b['age'], person_b['salary'], s=200, c='#e74c3c', label='B', zorder=3)
axes[0].scatter(person_c['age'], person_c['salary'], s=200, c='#2ecc71', label='C', zorder=3)
axes[0].set_xlabel('Age (years)')
axes[0].set_ylabel('Salary (₪)')
axes[0].set_title('Without Normalization\n(Salary dominates!)', fontsize=12)
axes[0].legend()

# With normalization
axes[1].scatter(a_norm[0], a_norm[1], s=200, c='#3498db', label='A', zorder=3)
axes[1].scatter(b_norm[0], b_norm[1], s=200, c='#e74c3c', label='B', zorder=3)
axes[1].scatter(c_norm[0], c_norm[1], s=200, c='#2ecc71', label='C', zorder=3)
axes[1].set_xlabel('Age (normalized)')
axes[1].set_ylabel('Salary (normalized)')
axes[1].set_title('With Normalization (0-1)\n(Both features equal!)', fontsize=12)
axes[1].legend()
axes[1].set_xlim(-0.1, 1.1)
axes[1].set_ylim(-0.1, 1.1)

plt.tight_layout()
plt.show()

<div dir="rtl" style="text-align: right; background-color: #fff8e1; padding: 15px; border-radius: 10px; border-right: 4px solid #ffc107;">

### 🔑 כלל זהב:

**תמיד לנרמל נתונים לפני KNN!**

אלגוריתמים מבוססי מרחק (KNN, K-Means, SVM) רגישים לסקאלה.

</div>

<div dir="rtl" style="text-align: right;">

## 🏷️ מה עם משתנים קטגוריאליים?

KNN עובד רק עם **מספרים** - אז מה עושים עם משתנים כמו "עיר" או "צבע"?

</div>

<div dir="rtl" style="text-align: right; background-color: #fff3e0; padding: 20px; border-radius: 10px; border-right: 4px solid #ff9800;">

### ⚠️ טעות נפוצה: המרה למספרים רגילים

נניח יש לנו עמודת "עיר":
- תל אביב → 1
- ירושלים → 2
- חיפה → 3

**הבעיה:** עכשיו המודל חושב ש"חיפה" (3) רחוקה יותר מ"תל אביב" (1) מאשר "ירושלים" (2)!

אין שום סיבה שזה יהיה נכון.

</div>

<div dir="rtl" style="text-align: right; background-color: #e8f5e9; padding: 20px; border-radius: 10px; border-right: 4px solid #4caf50;">

### ✅ הפתרון: One-Hot Encoding

יוצרים עמודה **נפרדת** לכל קטגוריה:

| עיר | City_TelAviv | City_Jerusalem | City_Haifa |
|-----|--------------|----------------|------------|
| תל אביב | 1 | 0 | 0 |
| ירושלים | 0 | 1 | 0 |
| חיפה | 0 | 0 | 1 |

עכשיו כל הערים "שוות" מבחינת המרחק!

</div>

In [ ]:
#@title 🎨 **הדגמה** - לחצו ▶ להרצה (הקוד מוסתר - אין צורך להבין אותו)
# Example of One-Hot Encoding
example_df = pd.DataFrame({'city': ['Tel Aviv', 'Jerusalem', 'Haifa', 'Tel Aviv']})
print('Before One-Hot Encoding:')
display(example_df)

# Apply One-Hot Encoding
encoded = pd.get_dummies(example_df, columns=['city'], prefix='City')
print('\nAfter One-Hot Encoding:')
display(encoded)

<div dir="rtl" style="text-align: right; background-color: #fff8e1; padding: 15px; border-radius: 10px; border-right: 4px solid #ffc107;">

### 🔑 סיכום: הכנת נתונים ל-KNN

| סוג משתנה | פעולה |
|-----------|-------|
| **מספרי** | נרמול (StandardScaler / MinMax) |
| **קטגוריאלי** | One-Hot Encoding |
| **בינארי (0/1)** | אפשר להשאיר כמו שהוא |

</div>

<div dir="rtl" style="text-align: right;">

# חלק 4: חלוקה ל-Train/Test 🔀

---

</div>

<div dir="rtl" style="text-align: right; background-color: #e3f2fd; padding: 20px; border-radius: 10px; border-right: 4px solid #2196f3;">

## 📊 למה צריך לחלק את הנתונים?

**בעיה:** אם נאמן ונבדוק על אותם נתונים - המודל "יזכור" אותם!

**פתרון:** מחלקים את הנתונים:
- **Train (אימון):** 70-80% - ללמידה
- **Test (בדיקה):** 20-30% - לבדיקת ביצועים

### החוק החשוב:
> **המודל אף פעם לא רואה את נתוני הבדיקה בזמן האימון!**

</div>

In [ ]:
#@title 🎨 **הדגמה** - לחצו ▶ להרצה (הקוד מוסתר - אין צורך להבין אותו)
# Visual explanation of Train/Test split
fig, ax = plt.subplots(figsize=(12, 3))

# Draw the full dataset bar
ax.barh(0, 80, color='#3498db', label='Train (80%)', height=0.5)
ax.barh(0, 20, left=80, color='#e74c3c', label='Test (20%)', height=0.5)

# Annotations
ax.text(40, 0, 'TRAIN\n(for learning)', ha='center', va='center', fontsize=14, color='white', fontweight='bold')
ax.text(90, 0, 'TEST\n(for evaluation)', ha='center', va='center', fontsize=12, color='white', fontweight='bold')

ax.set_xlim(0, 100)
ax.set_ylim(-0.5, 0.5)
ax.set_xlabel('Percentage of Data')
ax.set_yticks([])
ax.set_title('Train/Test Split', fontsize=14)
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

<div dir="rtl" style="text-align: right;">

# חלק 5: דוגמה מלאה - סיווג פירות 🍎🍊

---

נבנה מודל KNN לסיווג פירות לפי משקל וקוטר.

</div>

In [ ]:
#@title 🎨 **הדגמה** - לחצו ▶ להרצה (הקוד מוסתר - אין צורך להבין אותו)
# Create fruit dataset - with some overlap for realistic accuracy
np.random.seed(42)
n = 60

# Apples: lighter, smaller - but with overlap
apple_weight = np.random.normal(150, 25, n)  # grams
apple_diameter = np.random.normal(7, 0.7, n)  # cm

# Oranges: heavier, larger - but with overlap
orange_weight = np.random.normal(175, 25, n)  # closer to apples!
orange_diameter = np.random.normal(7.8, 0.7, n)

# Combine into DataFrame
df = pd.DataFrame({
    'weight': np.concatenate([apple_weight, orange_weight]),
    'diameter': np.concatenate([apple_diameter, orange_diameter]),
    'fruit': ['apple'] * n + ['orange'] * n
})

print(f'Dataset: {len(df)} fruits')
print(f'\nSample:')
display(df.sample(5))

In [ ]:
#@title 🎨 **הדגמה** - לחצו ▶ להרצה (הקוד מוסתר - אין צורך להבין אותו)
# Visualize the data
plt.figure(figsize=(10, 6))

apples = df[df['fruit'] == 'apple']
oranges = df[df['fruit'] == 'orange']

plt.scatter(apples['weight'], apples['diameter'], c='#e74c3c', s=100, 
            label='Apple 🍎', alpha=0.7, edgecolor='white')
plt.scatter(oranges['weight'], oranges['diameter'], c='#f39c12', s=100, 
            label='Orange 🍊', alpha=0.7, edgecolor='white')

plt.xlabel('Weight (grams)')
plt.ylabel('Diameter (cm)')
plt.title('Fruit Classification Dataset')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

<div dir="rtl" style="text-align: right;">

## שלב 1: הכנת הנתונים

</div>

In [ ]:
#@title ▶️ **הריצו תא זה**
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Separate features (X) and target (y)
X = df[['weight', 'diameter']]
y = df['fruit']

print('Features (X):')
print(X.shape)
print('\nTarget (y):')
print(y.value_counts())

In [ ]:
#@title 🎨 **הדגמה** - לחצו ▶ להרצה (הקוד מוסתר - אין צורך להבין אותו)
# Split into Train and Test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,      # 20% for testing
    random_state=42,    # For reproducibility
    stratify=y          # Keep class proportions
)

print(f'Training set: {len(X_train)} samples')
print(f'Test set: {len(X_test)} samples')
print(f'\nClass distribution in train:')
print(y_train.value_counts())

In [ ]:
#@title 🎨 **הדגמה** - לחצו ▶ להרצה (הקוד מוסתר - אין צורך להבין אותו)
# Normalize the features
scaler = StandardScaler()

# IMPORTANT: Fit only on training data!
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # Use same parameters

print('Before scaling:')
print(f'  Weight: mean={X_train["weight"].mean():.1f}, std={X_train["weight"].std():.1f}')
print(f'  Diameter: mean={X_train["diameter"].mean():.2f}, std={X_train["diameter"].std():.2f}')

print('\nAfter scaling:')
print(f'  Weight: mean={X_train_scaled[:, 0].mean():.2f}, std={X_train_scaled[:, 0].std():.2f}')
print(f'  Diameter: mean={X_train_scaled[:, 1].mean():.2f}, std={X_train_scaled[:, 1].std():.2f}')

<div dir="rtl" style="text-align: right; background-color: #fff3e0; padding: 15px; border-radius: 10px; border-right: 4px solid #ff9800;">

### ⚠️ שימו לב!

- **`fit_transform`** על Train בלבד!
- **`transform`** על Test (עם אותם פרמטרים)

למה? כי אסור ל-Scaler "לראות" את נתוני הבדיקה.

</div>

<div dir="rtl" style="text-align: right;">

## שלב 2: אימון המודל

</div>

In [ ]:
#@title ▶️ **הריצו תא זה**
from sklearn.neighbors import KNeighborsClassifier

# Create and train the model
knn = KNeighborsClassifier(n_neighbors=5)  # K=5
knn.fit(X_train_scaled, y_train)

print('Model trained!')
print(f'K = {knn.n_neighbors}')

<div dir="rtl" style="text-align: right;">

## שלב 3: חיזוי והערכה

</div>

In [ ]:
#@title ▶️ **הריצו תא זה**
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Predict on test set
y_pred = knn.predict(X_test_scaled)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.1%}')

<div dir="rtl" style="text-align: right; background-color: #e8f5e9; padding: 15px; border-radius: 10px; border-right: 4px solid #4caf50;">

### 💡 על דיוק ריאלי

**שימו לב:** הדיוק הוא לא 100% - וזה **נורמלי ואפילו רצוי!**

**למה?**
- בעולם האמיתי יש חפיפה בין קטגוריות (תפוח גדול יכול להיות דומה לתפוז קטן)
- דיוק של 100% על נתוני בדיקה עשוי להצביע על **בעיה** (דליפת מידע, נתונים פשוטים מדי)
- דיוק של 85-95% בבעיות אמיתיות הוא לרוב תוצאה טובה מאוד

</div>

In [ ]:
#@title 🎨 **הדגמה** - לחצו ▶ להרצה (הקוד מוסתר - אין צורך להבין אותו)
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['apple', 'orange'],
            yticklabels=['apple', 'orange'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

print('Reading the matrix:')
print(f'  True Positives (Apple correctly classified): {cm[0,0]}')
print(f'  True Negatives (Orange correctly classified): {cm[1,1]}')
print(f'  False Positives (Orange misclassified as Apple): {cm[0,1]}')
print(f'  False Negatives (Apple misclassified as Orange): {cm[1,0]}')

<div dir="rtl" style="text-align: right;">

# חלק 6: בחירת K אופטימלי 🎯

---

איך בוחרים את ערך K הטוב ביותר? יש שתי שיטות נפוצות:
1. **בדיקת דיוק** על סט הבדיקה
2. **שיטת המרפק (Elbow Method)**

</div>

<div dir="rtl" style="text-align: right; background-color: #e3f2fd; padding: 20px; border-radius: 10px; border-right: 4px solid #2196f3;">

## 🤔 השפעת K

| K קטן (למשל 1) | K גדול (למשל 50) |
|----------------|------------------|
| רגיש לרעש | עמיד לרעש |
| גבול החלטה מסובך | גבול החלטה חלק |
| סיכון ל-**Overfitting** | סיכון ל-**Underfitting** |

### הפתרון: לנסות כמה ערכים ולבחור את הטוב ביותר!

</div>

In [ ]:
#@title 🎨 **הדגמה** - לחצו ▶ להרצה (הקוד מוסתר - אין צורך להבין אותו)
# Test different K values
k_values = range(1, 21)
train_scores = []
test_scores = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    
    train_scores.append(knn.score(X_train_scaled, y_train))
    test_scores.append(knn.score(X_test_scaled, y_test))

# Find best K
best_k = k_values[np.argmax(test_scores)]
best_score = max(test_scores)

print(f'Best K: {best_k}')
print(f'Best Test Accuracy: {best_score:.1%}')

In [ ]:
#@title 🎨 **הדגמה** - לחצו ▶ להרצה (הקוד מוסתר - אין צורך להבין אותו)
# Visualize K selection
plt.figure(figsize=(10, 6))

plt.plot(k_values, train_scores, 'b-o', label='Train Accuracy', linewidth=2)
plt.plot(k_values, test_scores, 'r-o', label='Test Accuracy', linewidth=2)
plt.axvline(best_k, color='green', linestyle='--', label=f'Best K = {best_k}')

plt.xlabel('K (Number of Neighbors)')
plt.ylabel('Accuracy')
plt.title('Finding the Optimal K')
plt.legend()
plt.xticks(k_values)
plt.grid(True, alpha=0.3)
plt.show()

<div dir="rtl" style="text-align: right; background-color: #e8f5e9; padding: 15px; border-radius: 10px; border-right: 4px solid #4caf50;">

### 💡 מה למדנו מהגרף?

- **K קטן מדי:** Train accuracy גבוהה, Test accuracy נמוכה → **Overfitting**
- **K גדול מדי:** שתי הדיוקים יורדים → **Underfitting**
- **K אופטימלי:** נקודה שבה Test accuracy הכי גבוהה
- **טיפ:** עדיף K אי-זוגי (למנוע תיקו בהצבעה)

</div>

<div dir="rtl" style="text-align: right;">

## 📐 שיטת המרפק (Elbow Method)

שיטה נוספת לבחירת K - מסתכלים על **שגיאת הבדיקה** (Error) כפונקציה של K.

</div>

In [ ]:
#@title 🎨 **הדגמה** - לחצו ▶ להרצה (הקוד מוסתר - אין צורך להבין אותו)
# Elbow Method - using error rate
error_rates = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    y_pred_k = knn.predict(X_test_scaled)
    # Error rate = 1 - accuracy
    error_rates.append(1 - accuracy_score(y_test, y_pred_k))

# Plot
plt.figure(figsize=(10, 6))
plt.plot(k_values, error_rates, 'r-o', linewidth=2, markersize=8)
plt.xlabel('K (Number of Neighbors)')
plt.ylabel('Error Rate (1 - Accuracy)')
plt.title('Elbow Method - Finding Optimal K')
plt.xticks(k_values)
plt.grid(True, alpha=0.3)
plt.show()

<div dir="rtl" style="text-align: right; background-color: #e8f5e9; padding: 15px; border-radius: 10px; border-right: 4px solid #4caf50;">

### 💡 שיטת המרפק - איך קוראים את הגרף?

- מחפשים נקודה שבה הירידה בשגיאה **מתמתנת**
- הנקודה הזו נראית כמו "מרפק" בגרף
- אחרי ה"מרפק", הוספת שכנים לא משפרת משמעותית
- **הערה:** לא תמיד יש מרפק ברור - במקרה כזה משתמשים בשיטות אחרות

</div>

<div dir="rtl" style="text-align: right;">

# חלק 7: גבול ההחלטה (Decision Boundary) 🗺️

---

נראה איך K משפיע על "המפה" של ההחלטות:

</div>

In [ ]:
#@title 🎨 **הדגמה** - לחצו ▶ להרצה (הקוד מוסתר - אין צורך להבין אותו)
# Create decision boundary visualization
def plot_decision_boundary(X, y, k, ax, title):
    # Create mesh
    h = 0.02
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    
    # Train and predict
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X, y)
    Z = knn.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = np.array([0 if z == 'apple' else 1 for z in Z])
    Z = Z.reshape(xx.shape)
    
    # Plot
    cmap_light = ListedColormap(['#FFAAAA', '#AAFFAA'])
    ax.contourf(xx, yy, Z, cmap=cmap_light, alpha=0.6)
    
    # Plot points
    colors = ['#e74c3c' if label == 'apple' else '#f39c12' for label in y]
    ax.scatter(X[:, 0], X[:, 1], c=colors, edgecolor='black', s=50)
    
    ax.set_title(f'{title}\nK = {k}')
    ax.set_xlabel('Weight (scaled)')
    ax.set_ylabel('Diameter (scaled)')

# Compare different K values
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, k in zip(axes, [1, 5, 15]):
    plot_decision_boundary(X_train_scaled, y_train.values, k, ax, 
                          'Complex' if k == 1 else 'Balanced' if k == 5 else 'Smooth')

plt.tight_layout()
plt.show()

<div dir="rtl" style="text-align: right; background-color: #e8f5e9; padding: 15px; border-radius: 10px; border-right: 4px solid #4caf50;">

### 💡 מה למדנו מהגרפים?

- **K=1:** גבול מסובך מאוד, עוקב אחרי כל נקודה → Overfitting
- **K=5:** גבול מאוזן, לא רגיש לנקודות בודדות
- **K=15:** גבול חלק מאוד, אולי מפספס דפוסים מורכבים

</div>

<div dir="rtl" style="text-align: right;">

# חלק 8: חיזוי על נתון חדש 🔮

---

</div>

In [ ]:
#@title 🎨 **הדגמה** - לחצו ▶ להרצה (הקוד מוסתר - אין צורך להבין אותו)
# Train final model with best K
final_model = KNeighborsClassifier(n_neighbors=best_k)
final_model.fit(X_train_scaled, y_train)

# New fruit to classify
new_fruit = pd.DataFrame({'weight': [170], 'diameter': [7.8]})
print('New fruit:')
print(f'  Weight: {new_fruit["weight"].values[0]} grams')
print(f'  Diameter: {new_fruit["diameter"].values[0]} cm')

# Scale the new data (using the same scaler!)
new_fruit_scaled = scaler.transform(new_fruit)

# Predict
prediction = final_model.predict(new_fruit_scaled)
proba = final_model.predict_proba(new_fruit_scaled)

print(f'\nPrediction: {prediction[0]} {"🍎" if prediction[0] == "apple" else "🍊"}')
print(f'Confidence: Apple={proba[0][0]:.1%}, Orange={proba[0][1]:.1%}')

<div dir="rtl" style="text-align: right; background-color: #fff8e1; padding: 20px; border-radius: 10px;">

# 📝 סיכום - תהליך העבודה עם KNN

| שלב | פעולה | פקודה |
|-----|-------|-------|
| 1 | הפרדת מאפיינים ויעד | `X = df[features]`, `y = df[target]` |
| 2 | חלוקה ל-Train/Test | `train_test_split(X, y, test_size=0.2)` |
| 3 | נרמול | `scaler.fit_transform(X_train)`, `scaler.transform(X_test)` |
| 4 | יצירת מודל | `KNeighborsClassifier(n_neighbors=k)` |
| 5 | אימון | `model.fit(X_train, y_train)` |
| 6 | חיזוי | `model.predict(X_test)` |
| 7 | הערכה | `accuracy_score(y_test, y_pred)` |

</div>

<div dir="rtl" style="text-align: right; background-color: #e8f5e9; padding: 20px; border-radius: 10px; border-right: 4px solid #4caf50;">

## 🔑 נקודות חשובות לזכור

1. **תמיד לנרמל** - KNN רגיש לסקאלה!
2. **fit רק על Train** - הנרמול לא "רואה" את הבדיקה
3. **לבחור K בזהירות** - לא קטן מדי ולא גדול מדי
4. **K אי-זוגי** - למנוע תיקו
5. **להסתכל על Train ו-Test** - לזהות Overfitting

</div>

<div dir="rtl" style="text-align: right; background-color: #fff3e0; padding: 20px; border-radius: 10px; border-right: 4px solid #ff9800;">

## 🤔 שאלת הבנה - הרעיון של KNN

**נקודה חדשה צריכה להיות מסווגת. יש לה 5 שכנים קרובים:**
- 3 שכנים מסוג A
- 2 שכנים מסוג B

**שאלות:**
1. לאיזו קטגוריה תסווג הנקודה?
2. אם K=3 (רק 3 שכנים קרובים), והם 2 מסוג A ו-1 מסוג B - מה התוצאה?
3. למה עדיף K אי-זוגי?

</div>

<div dir="rtl" style="text-align: right; background-color: #e8f5e9; padding: 15px; border-radius: 10px;">

**תשובות:**
1. קטגוריה A (רוב של 3 מתוך 5)
2. קטגוריה A (רוב של 2 מתוך 3)
3. כדי למנוע תיקו (למשל 2 נגד 2 ב-K=4)

</div>

<div dir="rtl" style="text-align: right; background-color: #e8f5e9; padding: 20px; border-radius: 10px; border-right: 4px solid #4caf50;">

## ✍️ תרגיל 1: חישוב מרחק

**נתונות 3 נקודות:**
- A: (2, 3)
- B: (5, 7)
- C: (1, 1)

**חשבו:**
1. מה המרחק מ-A ל-B?
2. מה המרחק מ-A ל-C?
3. איזו נקודה קרובה יותר ל-A?

</div>

In [ ]:
#@title ✍️ **תרגול** - כתבו קוד!
# תרגיל 1 - חשבו את המרחקים
import numpy as np

A = (2, 3)
B = (5, 7)
C = (1, 1)

# חשבו את המרחקים
# dist_AB = ...
# dist_AC = ...

# הדפיסו את התוצאות

In [ ]:
#@title ✍️ **תרגול** - כתבו קוד!
# פתרון תרגיל 1
dist_AB = np.sqrt((5-2)**2 + (7-3)**2)
dist_AC = np.sqrt((1-2)**2 + (1-3)**2)

print(f'Distance A→B: √((5-2)² + (7-3)²) = √(9+16) = √25 = {dist_AB}')
print(f'Distance A→C: √((1-2)² + (1-3)²) = √(1+4) = √5 = {dist_AC:.2f}')
print(f'\nC קרובה יותר ל-A ({dist_AC:.2f} < {dist_AB})')

<div dir="rtl" style="text-align: right; background-color: #fff3e0; padding: 20px; border-radius: 10px; border-right: 4px solid #ff9800;">

## 🤔 שאלת הבנה - נרמול

**יש לנו 2 מאפיינים:**
- גובה באנשים: 150-200 ס"מ
- משקל: 50-100 ק"ג

**שאלות:**
1. אם לא נעשה נרמול, איזה מאפיין ישפיע יותר על המרחק?
2. אחרי Min-Max נרמול, מה יהיה הטווח של שני המאפיינים?
3. למה חשוב לעשות `fit` רק על Train?

</div>

<div dir="rtl" style="text-align: right; background-color: #e8f5e9; padding: 15px; border-radius: 10px;">

**תשובות:**
1. גובה (טווח 50 לעומת 50, אבל הערכים עצמם גבוהים יותר - 150-200 לעומת 50-100)
2. שניהם יהיו בטווח 0-1
3. כדי שה-Scaler לא "יראה" מידע מנתוני הבדיקה (data leakage)

</div>

<div dir="rtl" style="text-align: right; background-color: #e8f5e9; padding: 20px; border-radius: 10px; border-right: 4px solid #4caf50;">

## ✍️ תרגיל 2: בניית מודל KNN

**באמצעות הנתונים הבאים, בנו מודל KNN:**

```python
# נתוני אימון
X_train = [[1, 2], [2, 3], [3, 1], [6, 5], [7, 7], [8, 6]]
y_train = ['A', 'A', 'A', 'B', 'B', 'B']
```

**משימות:**
1. צרו מודל KNN עם K=3
2. אמנו את המודל
3. חזו את הקטגוריה של הנקודה [4, 4]
4. מה ההסתברות לכל קטגוריה?

</div>

In [ ]:
#@title ✍️ **תרגול** - כתבו קוד!
# תרגיל 2 - כתבו את הקוד שלכם
from sklearn.neighbors import KNeighborsClassifier

X_train = [[1, 2], [2, 3], [3, 1], [6, 5], [7, 7], [8, 6]]
y_train = ['A', 'A', 'A', 'B', 'B', 'B']

# 1. צרו מודל
# knn = ...

# 2. אמנו
# knn.fit(...)

# 3. חזו
# prediction = ...

# 4. הסתברויות
# proba = ...

In [ ]:
#@title ✍️ **תרגול** - כתבו קוד!
# פתרון תרגיל 2
from sklearn.neighbors import KNeighborsClassifier

X_train = [[1, 2], [2, 3], [3, 1], [6, 5], [7, 7], [8, 6]]
y_train = ['A', 'A', 'A', 'B', 'B', 'B']

# 1. יצירת מודל
knn = KNeighborsClassifier(n_neighbors=3)

# 2. אימון
knn.fit(X_train, y_train)

# 3. חיזוי
new_point = [[4, 4]]
prediction = knn.predict(new_point)
print(f'Prediction for [4,4]: {prediction[0]}')

# 4. הסתברויות
proba = knn.predict_proba(new_point)
print(f'Probabilities: A={proba[0][0]:.1%}, B={proba[0][1]:.1%}')

<div dir="rtl" style="text-align: right; background-color: #fff3e0; padding: 20px; border-radius: 10px; border-right: 4px solid #ff9800;">

## 🤔 שאלת הבנה - בחירת K

**נתון הגרף הבא של Train vs Test Accuracy:**

| K | Train Acc | Test Acc |
|---|-----------|----------|
| 1 | 100% | 75% |
| 3 | 95% | 85% |
| 5 | 90% | 88% |
| 10 | 85% | 86% |
| 20 | 80% | 82% |

**שאלות:**
1. מה הסימן ל-Overfitting בטבלה?
2. איזה K הכי טוב ולמה?
3. מה קורה כש-K גדול מדי?

</div>

<div dir="rtl" style="text-align: right; background-color: #e8f5e9; padding: 15px; border-radius: 10px;">

**תשובות:**
1. K=1: Train=100% אבל Test=75% - פער גדול = Overfitting
2. K=5: Test Accuracy הכי גבוהה (88%) עם פער קטן מ-Train
3. K גדול מדי (20): שתי הדיוקים יורדים = Underfitting

</div>

<div dir="rtl" style="text-align: right; background-color: #e8f5e9; padding: 20px; border-radius: 10px; border-right: 4px solid #4caf50;">

## 🎯 תרגיל מסכם: תהליך עבודה מלא

**נתון מערך נתונים של פרחים (Iris):**

```python
from sklearn.datasets import load_iris
iris = load_iris()
X = iris.data  # 4 מאפיינים
y = iris.target  # 3 סוגי פרחים
```

**בצעו את כל התהליך:**
1. חלקו ל-Train/Test (80/20)
2. נרמלו את הנתונים
3. מצאו את K האופטימלי (בדקו 1-15)
4. אמנו מודל סופי ודווחו על הדיוק

</div>

In [ ]:
#@title ✍️ **תרגול** - כתבו קוד!
# תרגיל מסכם - כתבו את הקוד שלכם
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# טעינת נתונים
iris = load_iris()
X = iris.data
y = iris.target

# 1. חלוקה
# X_train, X_test, y_train, y_test = ...

# 2. נרמול
# scaler = ...

# 3. מציאת K אופטימלי
# ...

# 4. מודל סופי
# ...

In [ ]:
#@title ✍️ **תרגול** - כתבו קוד!
# פתרון תרגיל מסכם
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
import numpy as np

# טעינת נתונים
iris = load_iris()
X = iris.data
y = iris.target

# 1. חלוקה
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')

# 2. נרמול
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. מציאת K אופטימלי
best_k = 1
best_score = 0
for k in range(1, 16):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    score = knn.score(X_test_scaled, y_test)
    if score > best_score:
        best_score = score
        best_k = k
print(f'\nBest K: {best_k} (accuracy: {best_score:.1%})')

# 4. מודל סופי
final_model = KNeighborsClassifier(n_neighbors=best_k)
final_model.fit(X_train_scaled, y_train)
y_pred = final_model.predict(X_test_scaled)
print(f'Final Accuracy: {accuracy_score(y_test, y_pred):.1%}')

<div dir="rtl" style="text-align: right; background-color: #e3f2fd; padding: 15px; border-radius: 10px;">

## 👀 בשיעור הבא...

**מחברת 7: מדדי ביצוע**

נלמד מדדים נוספים מעבר ל-Accuracy:
- Precision, Recall, F1-Score
- Confusion Matrix מפורטת
- מתי להשתמש בכל מדד

</div>